Import Python modules

In [1]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(font_scale=1.0, style='ticks', palette='colorblind')

Read in metadata

In [2]:
# Read in data
meta_df = pd.read_csv('../results/combined_metadata_with_host_groups.csv')

# Remove "A / " from subtype entries
meta_df['subtype'] = meta_df['subtype'].apply(lambda x: x[4:])
print(len(meta_df))
meta_df.head()

550503


,isolate_id,isolate_name,subtype,clade,passage_history,location,host,collection_date,host_group
0,EPI_ISL_20082926,A/turkey/North Carolina/25-003214-003/2025,H1N1,unassigned,Original,North America / United States / North Carolina,Turkey,2025-01-23,avian
1,EPI_ISL_20082915,A/turkey/North Carolina/25-000484-001/2024,H1N1,unassigned,Original,North America / United States / North Carolina,Turkey,2024-12-30,avian
2,EPI_ISL_20082903,A/turkey/Minnesota/25-011577-001/2025,H1N1,6B.1,Original,North America / United States / Minnesota,Turkey,2025-03-31,avian
3,EPI_ISL_20081192,A/swine/Shandong/2526/2022,H1N1,unassigned,NaN,Asia / China / Shandong,Swine,2022-11-15,swine
4,EPI_ISL_20080935,A/swine/Hebei/1231/2021,H1N1,unassigned,NaN,Asia / China / Hebei Province / Hebei,Swine,2021-12-31,swine


Make a table showing the distribution of isolates by host group.

In [3]:
host_counts = meta_df['host_group'].value_counts().to_frame(name='number of isolates')
host_counts.index.name = 'host'

print(host_counts.to_latex(
    caption='Distribution of isolates by host group.',
    label='tab:host_isolates',
    column_format='lr',
    formatters={'number of isolates': '{:,}'.format}
))

\begin{table}
\caption{Distribution of isolates by host group.}
\label{tab:host_isolates}
\begin{tabular}{lr}
\toprule
 & number of isolates \\
host &  \\
\midrule
human & 422,767 \\
avian & 85,019 \\
swine & 28,594 \\
other & 10,154 \\
bovine & 3,969 \\
\bottomrule
\end{tabular}
\end{table}



Make a table showing the distribution of isolates by subtype.

In [4]:
subtype_counts = meta_df['subtype'].value_counts().head(10).to_frame(name='Number of Isolates')
subtype_counts.index.name = 'Subtype'

print(subtype_counts.to_latex(
    column_format='lr',
    formatters={'Number of Isolates': '{:,}'.format},
    caption='Distribution of isolates by subtype (top 10).',
    label='tab:subtype_isolates',
    position='ht'
))

\begin{table}[ht]
\caption{Distribution of isolates by subtype (top 10).}
\label{tab:subtype_isolates}
\begin{tabular}{lr}
\toprule
 & Number of Isolates \\
Subtype &  \\
\midrule
H3N2 & 226,173 \\
H1N1 & 216,388 \\
H5N1 & 38,663 \\
H9N2 & 18,814 \\
H1N2 & 9,723 \\
H3N8 & 5,217 \\
H5N8 & 4,785 \\
H7N9 & 3,216 \\
H5N6 & 3,211 \\
H4N6 & 2,772 \\
\bottomrule
\end{tabular}
\end{table}



For each host, get the top 10 most common subtypes isolated from that host and report the fraction of that subtype from that host.

In [5]:
# For each host, get top 10 subtypes and fraction of that subtype from this host
results = []
hosts = ['human', 'avian', 'swine', 'bovine']
for host in hosts:
    # Get top 10 subtypes for this host
    host_df = meta_df[meta_df['host_group'] == host]
    top_subtypes = host_df['subtype'].value_counts().head(10)
    
    for subtype, count_from_host in top_subtypes.items():
        # Total count of this subtype across all hosts
        total_subtype_count = len(meta_df[meta_df['subtype'] == subtype])
        fraction_from_host = count_from_host / total_subtype_count
        
        results.append({
            'Host': host,
            'Subtype': subtype,
            'Count from Host': count_from_host,
            'Total Subtype Count': total_subtype_count,
            'Fraction from Host': fraction_from_host
        })

results_df = pd.DataFrame(results)

# Output as LaTeX tables
for host in hosts:
    host_data = results_df[results_df['Host'] == host].copy()
    host_data = host_data[['Subtype', 'Count from Host', 'Total Subtype Count', 'Fraction from Host']]
    
    print(host_data.to_latex(
        index=False,
        caption=f'Top 10 subtypes for {host} isolates with fraction of subtype from {host} hosts.',
        label=f'tab:{host}_subtypes',
        column_format='lrrr',
        formatters={
            'Count from Host': '{:,}'.format,
            'Total Subtype Count': '{:,}'.format,
            'Fraction from Host': '{:.3f}'.format
        }
    ))
    print()

\begin{table}
\caption{Top 10 subtypes for human isolates with fraction of subtype from human hosts.}
\label{tab:human_subtypes}
\begin{tabular}{lrrr}
\toprule
Subtype & Count from Host & Total Subtype Count & Fraction from Host \\
\midrule
H3N2 & 220,947 & 226,173 & 0.977 \\
H1N1 & 198,698 & 216,388 & 0.918 \\
H7N9 & 1,521 & 3,216 & 0.473 \\
H5N1 & 882 & 38,663 & 0.023 \\
H1N2 & 244 & 9,723 & 0.025 \\
H2N2 & 166 & 444 & 0.374 \\
H9N2 & 128 & 18,814 & 0.007 \\
H5N6 & 65 & 3,211 & 0.020 \\
H7N7 & 58 & 1,496 & 0.039 \\
H10N3 & 14 & 303 & 0.046 \\
\bottomrule
\end{tabular}
\end{table}


\begin{table}
\caption{Top 10 subtypes for avian isolates with fraction of subtype from avian hosts.}
\label{tab:avian_subtypes}
\begin{tabular}{lrrr}
\toprule
Subtype & Count from Host & Total Subtype Count & Fraction from Host \\
\midrule
H5N1 & 31,624 & 38,663 & 0.818 \\
H9N2 & 17,711 & 18,814 & 0.941 \\
H5N8 & 4,562 & 4,785 & 0.953 \\
H3N8 & 2,998 & 5,217 & 0.575 \\
H5N6 & 2,732 & 3,211 & 0.851 \\
H4N6